# PDF → Excel Converter
Converts **every PDF in this notebook's folder** into its own `.xlsx` file.

In [ ]:
%pip install pdfplumber openpyxl pypdf --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import pdfplumber
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Resolve the notebook's own folder ───────────────────────────────────────
try:
    BASE_DIR = Path(__file__).parent
except NameError:
    BASE_DIR = Path().resolve()


#Styling constants
HEADER_FONT     = Font(name="Arial", bold=True, color="FFFFFF", size=11)
HEADER_FILL     = PatternFill("solid", start_color="2F5597")
ALT_FILL        = PatternFill("solid", start_color="DCE6F1")
BODY_FONT       = Font(name="Arial", size=10)
CENTER          = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT            = Alignment(horizontal="left",   vertical="center", wrap_text=True)
THIN_SIDE       = Side(style="thin", color="B8CCE4")
THIN_BORDER     = Border(left=THIN_SIDE, right=THIN_SIDE, top=THIN_SIDE, bottom=THIN_SIDE)
TITLE_FONT      = Font(name="Arial", bold=True, size=14, color="1F3864")
PAGE_LABEL_FONT = Font(name="Arial", bold=True, size=11, color="FFFFFF")
PAGE_LABEL_FILL = PatternFill("solid", start_color="4472C4")
SECTION_FONT    = Font(name="Arial", bold=True, size=10, color="2F5597")


#Helpers
def style_header_row(ws, row_idx, col_count):
    for col in range(1, col_count + 1):
        cell = ws.cell(row=row_idx, column=col)
        cell.font, cell.fill, cell.alignment, cell.border = (
            HEADER_FONT, HEADER_FILL, CENTER, THIN_BORDER
        )


def style_data_row(ws, row_idx, col_count, alternate):
    for col in range(1, col_count + 1):
        cell = ws.cell(row=row_idx, column=col)
        cell.font      = BODY_FONT
        cell.fill      = ALT_FILL if alternate else PatternFill()
        cell.alignment = LEFT
        cell.border    = THIN_BORDER


def auto_fit_columns(ws, min_w=12, max_w=60):
    for col_cells in ws.columns:
        length = min_w
        for cell in col_cells:
            try:
                length = max(length, len(str(cell.value or "")) + 2)
            except Exception:
                pass
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(length, max_w)


#PDF extraction
def extract_pdf(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for num, page in enumerate(pdf.pages, start=1):
            tables     = page.extract_tables() or []
            text_lines = [
                ln.strip()
                for ln in (page.extract_text() or "").splitlines()
                if ln.strip()
            ]
            pages.append({"page_num": num, "tables": tables, "text_lines": text_lines})
    return pages


#Excel writing
def write_excel(pages, pdf_path, output_path):
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Content"
    ws.sheet_view.showGridLines = False

    row = 1

    #Doc title
    ws.cell(row=row, column=1, value=pdf_path.stem)
    ws.cell(row=row, column=1).font = TITLE_FONT
    ws.row_dimensions[row].height = 28
    row += 2

   
    for page in pages:

        
        ws.cell(row=row, column=1, value=f"  Page {page['page_num']}")
        ws.cell(row=row, column=1).font = PAGE_LABEL_FONT
        ws.cell(row=row, column=1).fill = PAGE_LABEL_FILL
        ws.cell(row=row, column=1).alignment = LEFT
        ws.row_dimensions[row].height = 18
        row += 2

        has_content = False

        # Tables
        for t_idx, table in enumerate(page["tables"], start=1):
            if not table:
                continue
            has_content = True

            if len(page["tables"]) > 1:
                ws.cell(row=row, column=1, value=f"Table {t_idx}").font = SECTION_FONT
                row += 1

            col_count = max(len(r) for r in table)
            for r_idx, trow in enumerate(table):
                padded = list(trow) + [None] * (col_count - len(trow))
                for c_idx, val in enumerate(padded, start=1):
                    ws.cell(row=row, column=c_idx, value=val or "")
                if r_idx == 0:
                    style_header_row(ws, row, col_count)
                else:
                    style_data_row(ws, row, col_count, alternate=(r_idx % 2 == 0))
                row += 1
            row += 1  # blank row after each table

        #Only shown when page has no tables, avoids duplication
        if not page["tables"] and page["text_lines"]:
            has_content = True
            for line in page["text_lines"]:
                c = ws.cell(row=row, column=1, value=line)
                c.font = BODY_FONT
                c.alignment = LEFT
                row += 1
            row += 1

        if not has_content:
            ws.cell(row=row, column=1,
                    value="(no extractable content on this page)").font = Font(
                name="Arial", italic=True, color="999999", size=9
            )
            row += 2

    auto_fit_columns(ws)
    wb.save(output_path)


#Main
pdf_files = sorted(BASE_DIR.glob("*.pdf"))

if not pdf_files:
    print(f"No PDF files found in: {BASE_DIR}")
    print("Place your PDFs in the same folder as this notebook and re-run.")
else:
    print(f"Found {len(pdf_files)} PDF(s) in: {BASE_DIR}\n")
    success, failed = 0, 0

    for pdf_path in pdf_files:
        output_path = BASE_DIR / (pdf_path.stem + ".xlsx")
        print(f"  Converting : {pdf_path.name}")
        print(f"  Output     : {output_path.name}")
        try:
            pages = extract_pdf(pdf_path)
            write_excel(pages, pdf_path, output_path)
            tables = sum(len(p['tables']) for p in pages)
            print(f"  ✓ Done  ({len(pages)} page(s), {tables} table(s))\n")
            success += 1
        except Exception as e:
            print(f"  ✗ Failed: {e}\n")
            failed += 1

    print("─" * 45)
    print(f"  Converted : {success}")
    print(f"  Failed    : {failed}")
    print(f"  Location  : {BASE_DIR}")

Found 7 PDF(s) in: C:\Users\MPL\Desktop\xcel

  Converting : 1.05.26  INVENTORY REPORT -UNIT-3 4th day Counting.pdf
  Output     : 1.05.26  INVENTORY REPORT -UNIT-3 4th day Counting.xlsx
  ✓ Done  (54 page(s), 54 table(s))

  Converting : 28 .04.26  INVENTORY REPORT -UNIT-3 -28-4-2026 (2).pdf
  Output     : 28 .04.26  INVENTORY REPORT -UNIT-3 -28-4-2026 (2).xlsx
  ✓ Done  (125 page(s), 125 table(s))

  Converting : 3.05.26 1.05.26  INVENTORY REPORT -UNIT-3 5th day Counting.pdf
  Output     : 3.05.26 1.05.26  INVENTORY REPORT -UNIT-3 5th day Counting.xlsx
  ✓ Done  (82 page(s), 82 table(s))

  Converting : 30 .04.26  INVENTORY REPORT -UNIT-3 3rd Counting (1).pdf
  Output     : 30 .04.26  INVENTORY REPORT -UNIT-3 3rd Counting (1).xlsx
  ✓ Done  (48 page(s), 48 table(s))

  Converting : 4-5-26 INVENTORY REPORT -UNIT-3 - 6th day.pdf
  Output     : 4-5-26 INVENTORY REPORT -UNIT-3 - 6th day.xlsx
  ✓ Done  (81 page(s), 81 table(s))

  Converting : 5-5-26  INVENTORY REPORT -UNIT-3 - 7tth day.p